In [5]:
pip install beautifulsoup4


Note: you may need to restart the kernel to use updated packages.


In [7]:
import requests

url = "https://www.economic.ntpc.gov.tw/News"

headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(url, headers=headers, verify=False)

print(res.status_code)
print(res.text[:200])

c:\Users\USER\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.economic.ntpc.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\USER\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.economic.ntpc.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200
<!DOCTYPE html>

<html lang="zh-Hant-tw" xml:lang="zh-Hant-tw">
<head>
    <meta name="viewport" content="width=device-width" />
    <meta http-equiv="X-UA-Compatible" content="IE=EmulateIE10">



In [8]:
import requests
from bs4 import BeautifulSoup

url = "https://www.economic.ntpc.gov.tw/News"

headers = {
    "User-Agent": "Mozilla/5.0"
}

res = requests.get(url, headers=headers, verify=False)

soup = BeautifulSoup(res.text, "html.parser")

# 抓新聞區塊（這裡要稍微觀察HTML）
titles = soup.select("a")

for t in titles:
    text = t.text.strip()
    link = t.get("href")

    if text and link:
        print(text, "https://www.economic.ntpc.gov.tw" + link)

c:\Users\USER\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.economic.ntpc.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\USER\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.economic.ntpc.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


跳到主要內容區 https://www.economic.ntpc.gov.tw#AC
::: https://www.economic.ntpc.gov.tw#U
網站導覽 https://www.economic.ntpc.gov.tw/Api/SideNavigation/Index
訂閱電子報 https://www.economic.ntpc.gov.tw/Api/NewsPaper/Index
前往Google(另開新視窗) https://www.economic.ntpc.gov.twhttps://www.google.com/search?q=site%3Awww.economic.ntpc.gov.tw
企業精典獎 https://www.economic.ntpc.gov.twhttps://www.economic.ntpc.gov.tw/Api/News/Page?id=8149
參展補助 https://www.economic.ntpc.gov.twhttps://www.economic.ntpc.gov.tw/Custom/111%E5%B9%B4%E6%96%B0%E5%8C%97%E5%B8%82%E6%94%BF%E5%BA%9C%E7%B6%93%E6%BF%9F%E7%99%BC%E5%B1%95%E5%B1%80%E9%BC%93%E5%8B%B5%E5%BB%A0%E5%95%86%E5%9C%8B%E5%85%A7%E5%A4%96%E5%8F%83%E5%B1%95%E8%A3%9C%E5%8A%A9%E8%A8%88%E7%95%AB
智慧微電網補助 https://www.economic.ntpc.gov.twhttps://www.economic.ntpc.gov.tw/Custom/115%E5%B9%B4%E5%BA%A6%E6%96%B0%E5%8C%97%E5%B8%82%E6%99%BA%E6%85%A7%E5%BE%AE%E9%9B%BB%E7%B6%B2%E7%A4%BA%E7%AF%84%E5%A0%B4%E5%9F%9F%E6%8E%A8%E5%BB%A3%E8%A3%9C%E5%8A%A9%E8%A8%88%E7%95%AB
關於本局 https://www.economic.ntp

In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import urllib3

# 關閉SSL警告
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

base_url = "https://www.economic.ntpc.gov.tw"

headers = {
    "User-Agent": "Mozilla/5.0"
}

def get_news_list(page):
    url = f"{base_url}/News?page={page}"
    res = requests.get(url, headers=headers, verify=False)
    soup = BeautifulSoup(res.text, "html.parser")

    news_list = []

    # 👉 抓所有連結（再過濾）
    for a in soup.select("a"):
        title = a.text.strip()
        link = a.get("href")

        # 👉 過濾真正新聞（關鍵）
        if link and "/News/" in link and len(title) > 5:
            full_link = base_url + link
            news_list.append((title, full_link))

    return news_list


def get_news_content(url):
    res = requests.get(url, headers=headers, verify=False)
    soup = BeautifulSoup(res.text, "html.parser")

    # 👉 抓日期（可能需要微調）
    date = ""
    date_tag = soup.find("time")
    if date_tag:
        date = date_tag.text.strip()

    # 👉 抓內文
    content = ""
    paragraphs = soup.select("p")
    for p in paragraphs:
        content += p.text.strip() + "\n"

    return date, content


# 🔥 主程式（自動翻頁）
all_data = []

for page in range(1, 4):  # 👉 先抓3頁（你可以改）
    print(f"抓第 {page} 頁...")

    news_list = get_news_list(page)

    for title, link in news_list:
        print("處理:", title)

        try:
            date, content = get_news_content(link)

            all_data.append({
                "標題": title,
                "日期": date,
                "連結": link,
                "內文": content
            })

            time.sleep(1)  # 防止被擋

        except Exception as e:
            print("錯誤:", e)

# 💾 存CSV
with open("ntpc_news_full.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["標題", "日期", "連結", "內文"])
    writer.writeheader()
    writer.writerows(all_data)

print("完成！已存成 ntpc_news_full.csv")

抓第 1 頁...
處理: 親子連假去哪玩？  新北招商案活動優惠陪你歡度兒童...


2026.04.01

【新北市訊】清明及兒童節連假即將到來，出遊行程準備好了嗎？新北市政府經發局特別彙整各大商業招商案推出的連假限定優惠與親子活動，邀請民眾把握假期走訪新北，享受豐富...
處理: 115年新北市群眾募資計畫徵件正式開跑！  導入A...


2026.03.31

【新北市訊】 115年度新北市群眾募資輔導計畫即日起開放徵件，面對AI驅動的全球數位轉型浪潮，如何協助企業品牌精準對接市場成為轉型關鍵！今年首度將AI工具應用納...
處理: 新北綠能治理再升級！攜手三方強化廉能機制  補助政...


2026.03.27

【新北市訊】為加速淨零轉型並強化綠能產業治理，新北市政府經濟發展局於今（27）日辦理「115年度再生能源補助政策與健全綠能發展說明會」，現場吸引眾多光電業者及能...
處理: 新北招商再下一城！  全美綠能注入鉅資投資土城 強...


2026.03.26

【新北市訊】新北招商成果再添一樁！全美綠能股份有限公司斥資約新臺幣20億元，於土城區興建地上13層、地下4層之現代化廠辦大樓，今（26）日舉辦開工動土典禮。本案...
處理: 親子出遊首選  新北觀光工廠推兒童節專屬優惠  邀...


2026.03.25

【新北市訊】兒童節連假即將到來，新北市政府邀請大小朋友來到新北特色觀光工廠，探索手作樂趣、體驗產業文化，享受寓教於樂的旅遊新體驗。今(115)年多家觀光工廠特別...
處理: 徵件倒數!「新北企業精典獎」徵件4/8截止


2026.03.20

【新北市訊】第三屆「新北企業精典獎」徵件進入最後倒數，將於115年4月8日截止受理報名。本屆獎項設有企業類「創新投資」、「多元服務」、「永續發展」、「寰宇國際」...
處理: 富致科技斥資8億興建五股營運總部 正式啟用  攜手...


2026.03.10

【新北市訊】新北招商一條龍再傳捷報！深耕電子零組件與精密製造領域的富致科技，斥資約8億元於五股區興建地上7層、地下2層之營運總部大樓暨自用廠房，今(10)日舉辦...
處理: 把握連假後黃金申請期！  新北企業參展補助延長收件...


2026.03.02

【新北市訊】為協助本市企業拓展國內外市場、強化品牌能見度，新北市政